# MedVision AI — Vue d'ensemble du projet

**Auteur :** Yann Smatti  
**Date :** Mars 2026  
**Stack :** TensorFlow 2.16 · MLflow 2.20 · DVC 3.59 · FastAPI · Streamlit

---

## Objectif de ce notebook

Ce notebook est le **point d'entrée unique** du repo. Il permet de :

1. Vérifier l'environnement Python et les dépendances installées
2. Charger et inspecter les configurations YAML de chaque track
3. Auditer le registre de modèles partagé (API + Streamlit)
4. Contrôler les manifests CSV et l'état des datasets
5. Faire le bilan des artefacts produits (modèles, overlays, rapports)
6. Consulter l'état des expériences MLflow

### Quatre tracks du projet

| # | Track | Classes | Architecture | Type |
|---|-------|---------|--------------|------|
| 1 | Chest X-ray classification | NORMAL / PNEUMONIA | EfficientNetB0 | Binaire |
| 2 | Brain MRI classification | glioma / meningioma / pituitary / no_tumor | EfficientNetB0 | Multi-classe |
| 3 | Brain tumor segmentation | 4 classes + masque | U-Net multitâche | Seg + Classif |
| 4 | Chest X-ray segmentation | NORMAL / PNEUMONIA + masque poumon | U-Net multitâche | Seg + Classif |

---

**Prérequis :** lancer ce notebook depuis la **racine du repo**.

In [ ]:
## 1. Vérification de l'environnement
from pathlib import Path
import sys
import importlib.metadata

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    raise RuntimeError("Lance ce notebook depuis la racine du repo medvision-ai")

print(f"Python        : {sys.version.split()[0]}")
print(f"Repo root     : {ROOT}")
print()

# Versions des dépendances clés
deps = ['tensorflow', 'mlflow', 'dvc', 'fastapi', 'streamlit', 'numpy', 'pandas', 'sklearn', 'PIL']
for dep in deps:
    try:
        pkg = 'scikit-learn' if dep == 'sklearn' else ('Pillow' if dep == 'PIL' else dep)
        ver = importlib.metadata.version(pkg)
        print(f"  {dep:<15} {ver}")
    except importlib.metadata.PackageNotFoundError:
        print(f"  {dep:<15} NON INSTALLÉ")


In [ ]:
## 2. Chargement des configurations YAML
from src.utils.config import load_config
import pandas as pd
import json

config_paths = {
    'global'            : 'configs/config.yaml',
    'brain_mri'         : 'configs/brain_tumor_mri.yaml',
    'brain_seg'         : 'configs/brain_tumor_segmentation.yaml',
    'chest_seg'         : 'configs/chest_xray_segmentation.yaml',
}

configs = {}
for name, path in config_paths.items():
    try:
        configs[name] = load_config(path)
        print(f"✓ {name:<20} chargé — {len(configs[name])} clés")
    except Exception as e:
        configs[name] = {}
        print(f"✗ {name:<20} ERREUR: {e}")

# Résumé tabulaire des paramètres clés
rows = []
for name, cfg in configs.items():
    rows.append({
        'config'      : name,
        'image_size'  : cfg.get('image_size', '—'),
        'batch_size'  : cfg.get('batch_size', '—'),
        'epochs'      : cfg.get('epochs', '—'),
        'num_classes' : cfg.get('num_classes', '—'),
        'backbone'    : cfg.get('backbone', cfg.get('base_model', '—')),
    })

pd.DataFrame(rows).set_index('config')


In [ ]:
## 3. Inspection du registre de modèles
try:
    from src.registry.model_registry import load_registry
    registry = load_registry()

    problems = registry.get('problems', {})
    print(f"Problems déclarés : {list(problems.keys())}\n")

    rows = []
    for problem_name, problem_cfg in problems.items():
        models_in_problem = problem_cfg.get('models', {})
        for model_id, model_cfg in models_in_problem.items():
            model_path = Path(model_cfg.get('path', ''))
            rows.append({
                'problem'    : problem_name,
                'model_id'   : model_id,
                'type'       : model_cfg.get('type', '—'),
                'path'       : str(model_path),
                'exists'     : '✓' if model_path.exists() else '✗',
            })

    if rows:
        df = pd.DataFrame(rows)
        print(df.to_string(index=False))
    else:
        print("Registre vide ou format inattendu.")
        print(json.dumps(registry, indent=2, default=str)[:1000])

except Exception as exc:
    print(f"Impossible de charger le registre : {exc}")
    print("Vérifie que src/registry/model_registry.py est présent et fonctionnel.")


In [ ]:
## 4. Inspection des manifests et datasets
manifest_paths = {
    'brain_seg'  : 'data/processed/brain_tumor_segmentation/manifest.csv',
    'chest_seg'  : 'data/processed/chest_xray_segmentation/manifest.csv',
}

for track, mp in manifest_paths.items():
    manifest = Path(mp)
    print(f"\n{'='*50}")
    print(f"Track : {track}  —  {mp}")
    if manifest.exists() and manifest.stat().st_size > 0:
        df = pd.read_csv(manifest)
        print(f"  Lignes totales : {len(df)}")
        if 'split' in df.columns:
            print(f"  Splits         : {dict(df['split'].value_counts())}")
        if 'label' in df.columns:
            print(f"  Labels         : {dict(df['label'].value_counts())}")
        # Vérification intégrité : fichiers réellement présents
        if 'image_path' in df.columns:
            present = df['image_path'].apply(lambda p: Path(p).exists()).sum()
            print(f"  Images vérif.  : {present}/{len(df)} fichiers présents sur disque")
        if 'mask_path' in df.columns:
            m_present = df['mask_path'].apply(lambda p: Path(p).exists()).sum()
            print(f"  Masques vérif. : {m_present}/{len(df)} fichiers présents sur disque")
    else:
        print("  ⚠ Manifest absent ou vide — lance le stage 'prepare' DVC")


In [ ]:
## 5. Inventaire des artefacts et données
import matplotlib.pyplot as plt
from collections import defaultdict

summary = {}
dirs_to_check = {
    'data/raw'                                          : 'raw data',
    'data/processed'                                    : 'processed data',
    'artifacts/models'                                  : 'modèles exportés',
    'artifacts/overlays'                                : 'overlays Grad-CAM',
    'artifacts/reports'                                 : 'rapports JSON',
    'mlruns'                                            : 'expériences MLflow',
}

rows = []
for rel, label in dirs_to_check.items():
    p = ROOT / rel
    n_files = sum(1 for f in p.rglob('*') if f.is_file()) if p.exists() else 0
    size_mb = sum(f.stat().st_size for f in p.rglob('*') if f.is_file()) / 1e6 if p.exists() else 0
    rows.append({'dossier': rel, 'label': label, 'fichiers': n_files, 'taille_mb': round(size_mb, 2), 'présent': p.exists()})

df_inv = pd.DataFrame(rows)
print(df_inv.to_string(index=False))
print()

# Graphe barres
fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#2ecc71' if r else '#e74c3c' for r in df_inv['présent']]
ax.barh(df_inv['label'], df_inv['fichiers'], color=colors)
ax.set_xlabel('Nombre de fichiers')
ax.set_title('Inventaire du repo MedVision AI')
for i, (n, s) in enumerate(zip(df_inv['fichiers'], df_inv['taille_mb'])):
    ax.text(n + 0.2, i, f'{n} ({s} MB)', va='center', fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
## 6. Expériences MLflow — bilan
import mlflow
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri('file:./mlruns')
client = MlflowClient()

experiments = client.search_experiments()
print(f"Expériences MLflow trouvées : {len(experiments)}\n")

rows = []
for exp in experiments:
    runs = client.search_runs(experiment_ids=[exp.experiment_id])
    if runs:
        best = sorted(runs, key=lambda r: r.info.start_time, reverse=True)[0]
        rows.append({
            'experiment'  : exp.name,
            'nb_runs'     : len(runs),
            'dernier_run' : best.info.run_id[:8],
            'status'      : best.info.status,
            'val_acc'     : round(best.data.metrics.get('val_accuracy', float('nan')), 4),
            'val_loss'    : round(best.data.metrics.get('val_loss', float('nan')), 4),
        })

if rows:
    df_mlflow = pd.DataFrame(rows)
    print(df_mlflow.to_string(index=False))
else:
    print("Aucun run MLflow enregistré. Lance un entraînement pour peupler mlruns/.")


In [ ]:
## 7. État du pipeline DVC
import subprocess, json

result = subprocess.run(
    ['dvc', 'status', '--json'],
    capture_output=True, text=True, cwd=str(ROOT)
)

if result.returncode == 0:
    try:
        status = json.loads(result.stdout)
    except json.JSONDecodeError:
        status = {}

    if not status:
        print("✅  Tous les stages DVC sont à jour.")
    else:
        print(f"⚠  {len(status)} stage(s) nécessitent une mise à jour :\n")
        for stage_name, details in status.items():
            if isinstance(details, list):
                print(f"  • {stage_name}")
                for change in details[:3]:
                    print(f"      {change}")
            else:
                print(f"  • {stage_name}: {details}")
else:
    print("DVC non disponible ou pipeline non initialisé.")
    print(result.stderr[:400] if result.stderr else "")
    print("\nPour initialiser : `dvc init` puis `dvc repro`")


## 8. Ordre de travail recommandé

Pour reproduire un run complet sur un nouveau poste :

```bash
# 1. Installer les dépendances
pip install -r requirements.txt

# 2. Télécharger les données (Kaggle)
powershell -ExecutionPolicy Bypass -File scripts/download_dataset.ps1

# 3. Préparer les manifests
dvc repro prepare_brain_seg prepare_chest_seg

# 4. Lancer l'entraînement
dvc repro train_classification_chest
dvc repro train_brain_mri
dvc repro train_seg_brain
dvc repro train_seg_chest

# 5. Consulter les métriques
mlflow ui --host 127.0.0.1 --port 5000

# 6. Lancer l'API + Streamlit
uvicorn src.api.main:app --reload &
streamlit run streamlit_app.py
```

### Points de contrôle critiques

| Étape | Indicateur de succès |
|-------|---------------------|
| Download | `data/raw/` non vide |
| Prepare | `manifest.csv` avec > 100 lignes |
| Train | Artefact `.keras` dans `artifacts/models/` |
| Evaluate | Run MLflow avec `val_accuracy` > 0.80 |
| API | `GET /health` → `{"status": "ok"}` |
| Streamlit | Prédiction stable sur une image test |

> Les notebooks `01` à `21` détaillent chaque étape de bout en bout.
